# <font color="steelblue">Detección de fraude sanitario</font>

**Material desarrollado por los [equipos de trabajo de IA4LEGOS](https://ia4legos.umh.es/)**

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a>

No olvides hacer una copia si deseas utilizarlo.

## <font color="steelblue">Objetivos del proyecto</font>

A partir de reclamaciones de seguros médicos, construir, comparar y **desplegar** un clasificador que detecte **fraude** (`Is_Fraud`). Este proyecto entrena una competencia que no aparece en los demás: la **ingeniería de variables y la codificación de categóricas de alta cardinalidad sin fuga**, en un problema con marco **operativo** (priorizar qué reclamaciones investigar). Tendréis que:

* **Transformar fechas** y crear variables derivadas con sentido de negocio.
* **Codificar categóricas de alta cardinalidad** (`diagnosis_code` ICD-10, `procedure_code` CPT, `state`, `provider_id`) sin que un *one-hot* explote ni se filtre el objetivo: **codificación por destino/frecuencia con validación cruzada interna**.
* Tratar el **desequilibrio** (el fraude es minoritario) y el **coste asimétrico** (no detectar un fraude vs investigar de más).
* Pensar en términos de **priorización**: ordenar por probabilidad y **revisar el top-K** (precision@K, curva de ganancia), como en el cuaderno de *Detección de anomalías*.


## <font color="steelblue">El conjunto de datos</font>

### <font color="steelblue">Origen y estructura</font>

Se trata de un conjunto de **10.000 reclamaciones** (*claims*) presentadas a una aseguradora sanitaria, descritas por 15 variables y un indicador binario de fraude. Los datos son **sintéticos pero realistas**: fueron generados imitando la estructura, las escalas y las correlaciones de un registro real de facturación médica. Esto tiene una consecuencia que conviene interiorizar desde el principio: los **patrones de fraude fueron inyectados por diseño**. En concreto, el fraude tiende a asociarse con **importes más altos**, **plazos de presentación más cortos** y se **concentra en ciertos proveedores**; además, se introdujeron **valores faltantes a propósito** para forzar un preprocesado realista.

Cada fila es una **reclamación individual** —no un paciente ni un proveedor—, y las variables describen cuatro perspectivas distintas de esa reclamación: sus **importes**, el **paciente** que recibió la asistencia, el **proveedor** que la facturó y la **codificación médica** del acto asistencial.

### <font color="steelblue">Diccionario de variables</font>

**Bloque 1 — Reclamación y finanzas**

| Variable | Tipo | Descripción |
|---|---|---|
| `claim_amount` | Numérica continua | **Importe reclamado** por el proveedor. Sigue una distribución **log-normal**: fuertemente asimétrica a la derecha, con una cola larga de importes muy elevados. Los modelos lineales se benefician de una transformación logarítmica; los árboles son indiferentes. |
| `approved_amount` | Numérica continua | **Importe finalmente aprobado** por la aseguradora. Suele ser menor o igual que `claim_amount`; su cociente es un indicador natural de discrepancia. |
| `claim_status` | Nominal | Estado de la reclamación: `approved` (aprobada), `rejected` (rechazada) o `pending` (pendiente de resolución). |
| `insurance_type` | Nominal | Régimen de aseguramiento: `private` (seguro privado), `Medicare` (programa público para mayores de 65 años y ciertas discapacidades), `Medicaid` (programa público para rentas bajas) o `self-pay` (paciente sin seguro que paga de su bolsillo). |

**Bloque 2 — Paciente**

| Variable | Tipo | Descripción |
|---|---|---|
| `patient_age` | Numérica | **Edad** del paciente, en años. |
| `patient_gender` | Nominal | **Sexo** del paciente. |
| `state` | Nominal | **Estado** de EE. UU. donde se prestó la asistencia. Cardinalidad media-alta (hasta 50 categorías). |
| `chronic_condition` | Binaria (0/1) | Indica si el paciente padece alguna **enfermedad crónica**. |
| `prior_visits_12m` | Numérica de recuento | Número de **visitas previas** del paciente en los últimos 12 meses. Aproxima su intensidad de uso del sistema. |

**Bloque 3 — Proveedor**

| Variable | Tipo | Descripción |
|---|---|---|
| `provider_id` | Nominal, **alta cardinalidad** | **Identificador** del médico o centro que emite la factura. Formalmente es un identificador, pero —como el fraude se concentra en ciertos proveedores— **contiene señal real**. Es la variable más delicada del conjunto (véase la advertencia 4). |
| `provider_specialty` | Nominal | **Especialidad médica** del proveedor (cardiología, radiología, etc.). |
| `monthly_claim_volume` | Numérica | **Volumen mensual de reclamaciones** emitidas por ese proveedor. Es un atributo **del proveedor**, no de la reclamación: se **repite idéntico** en todas las filas del mismo proveedor. |

**Bloque 4 — Codificación médica** *(ambas de alta cardinalidad)*

| Variable | Tipo | Descripción |
|---|---|---|
| `diagnosis_code` | Nominal, alta cardinalidad | Código **ICD-10** del diagnóstico (una letra seguida de dígitos, p. ej. `E11.9` para diabetes tipo 2). La **primera letra** identifica el **capítulo** de la clasificación, lo que permite agrupar miles de códigos en una veintena de categorías. |
| `procedure_code` | Nominal, alta cardinalidad | Código **CPT** del procedimiento realizado (cinco dígitos). Sus **rangos numéricos** delimitan secciones (cirugía, radiología, laboratorio, etc.), lo que ofrece igualmente una vía de agrupación. |

**Bloque 5 — Variables temporales y adicionales**

| Variable | Tipo | Descripción |
|---|---|---|
| `claim_submission_date` | Fecha | **Fecha de presentación** de la reclamación. |
| `days_to_submission` | Numérica | **Días transcurridos** entre la prestación del servicio y su facturación. Por diseño, el fraude tiende a presentarse **más rápido**. |
| `length_of_stay` | Numérica | **Días de estancia** hospitalaria. Solo tiene sentido en ingresos: en las visitas ambulatorias o de urgencias es estructuralmente inaplicable. |
| `visit_type` | Nominal | Tipo de asistencia: `outpatient` (ambulatoria), `inpatient` (con ingreso) o `emergency` (urgencias). |

**Variable objetivo**

| Variable | Valores | Descripción |
|---|---|---|
| `Is_Fraud` | 0 / 1 | Indica si la reclamación es **fraudulenta**. Es la clase minoritaria: conviene comprobar su proporción con `value_counts(normalize=True)` antes de elegir métrica alguna. |

### <font color="steelblue">Advertencias metodológicas</font>

1. **Fuga de información: define primero *cuándo* predices.** `claim_status` y `approved_amount` solo se conocen **después de adjudicar** la reclamación. Si el objetivo es detectar el fraude **antes de pagar**, ambas son información del futuro y su uso constituye una **fuga**: el modelo obtendrá métricas excelentes y será inservible en producción, porque en el momento de decidir esa información aún no existe. La regla general es simple y conviene enunciarla así: **una variable es un predictor legítimo solo si estará disponible en el instante en que se necesite la predicción.** Decidid ese instante y justificad qué variables sobreviven.

2. **El desequilibrio hace inútil la exactitud.** El fraude es, por naturaleza, un suceso raro. Un modelo que prediga siempre «legítima» acertará en la inmensa mayoría de los casos sin haber aprendido nada. Las métricas adecuadas son el **recall** de la clase fraudulenta, la **precisión a un umbral operativo** y el **AUC de la curva precisión-recall** (no el AUC-ROC, que resulta optimista con clases muy desequilibradas).

3. **Los costes son asimétricos, y el umbral debe reflejarlo.** Un falso negativo significa **pagar un fraude**; un falso positivo, **investigar una reclamación legítima**, con el coste administrativo y el daño a la relación con el proveedor que ello implica. El umbral de decisión no debe fijarse en 0,5 por inercia, sino elegirse **minimizando el coste esperado**, una vez estimado el coste relativo de ambos errores.

4. **`provider_id`: la variable más peligrosa.** Como el fraude se concentra en ciertos proveedores, incluirla mejora las métricas de forma llamativa. Pero el modelo estará **memorizando qué proveedores defraudaron**, no aprendiendo **cómo es una reclamación fraudulenta**, y será incapaz de detectar a un defraudador nuevo. Además, si un mismo proveedor aparece en entrenamiento y en test, la partición aleatoria **filtra** su comportamiento entre ambos conjuntos. Si el objetivo es generalizar a proveedores no vistos, la partición debe hacerse por **grupos** (`GroupKFold` agrupando por `provider_id`). La misma cautela alcanza a `monthly_claim_volume`, que es un atributo del proveedor repetido en todas sus filas.

5. **Alta cardinalidad: cómo tratar `diagnosis_code` y `procedure_code`.** Una codificación *one-hot* generaría miles de columnas dispersas. Las tres salidas razonables son: **agrupar** por capítulo ICD-10 o sección CPT (la más interpretable); usar **codificación por objetivo** (*target encoding*), que **debe calcularse dentro de cada partición** de la validación cruzada o volverá a producir fuga; o emplear **CatBoost**, que las trata de forma nativa con su codificación ordenada, precisamente diseñada para este problema.

6. **Estructura temporal.** Los datos llevan fecha, y el fraude **evoluciona**: los defraudadores cambian de táctica. Una partición **aleatoria** mezcla pasado y futuro y sobreestima el rendimiento. La evaluación realista consiste en entrenar con las reclamaciones **más antiguas** y validar con las **más recientes**. Nótese además que `claim_submission_date` y `days_to_submission` son **redundantes** en parte; la segunda ya extrae la información útil de la primera.

7. **Faltantes con distinta naturaleza.** `length_of_stay` no falta al azar: **no aplica** a las visitas ambulatorias o de urgencias, exactamente igual que el embarazo no aplica a los hombres. Imputar una mediana ahí es un error conceptual; conviene codificar la ausencia como categoría propia o restringir el análisis. El resto de faltantes, introducidos artificialmente, sí pueden imputarse; verificad antes con `df.isna().groupby(df['Is_Fraud']).mean()` si la ausencia está **asociada al fraude**, en cuyo caso la propia ausencia es información.

8. **Un dato sintético no descubre nada.** Los patrones que el modelo encuentre (importes altos, plazos cortos, ciertos proveedores) serán exactamente los que se **inyectaron al generar los datos**. La interpretación con SHAP es un excelente ejercicio de **verificación** —¿recupera el modelo los patrones plantados?—, pero **no un hallazgo** sobre el fraude sanitario real.

9. **Consideraciones éticas.** `patient_gender`, `patient_age`, `state` e `insurance_type` (que actúa como aproximación de nivel socioeconómico: `Medicaid` ↔ rentas bajas, `self-pay` ↔ sin cobertura) son **atributos sensibles o correlacionados con ellos**. Un sistema de detección de fraude que penalice sistemáticamente a los pacientes de un régimen o de una región produce un **impacto discriminatorio**, aunque nunca use explícitamente el atributo protegido. Merece la pena examinar el rendimiento **por subgrupos**, no solo el global.

## <font color="steelblue">Reglas del juego (buenas prácticas obligatorias)</font>

1. **Codifica según la cardinalidad y SIN fuga.** Baja cardinalidad → one-hot (agrupando categorías raras); **alta cardinalidad** (`diagnosis_code`, `procedure_code`, `state`, `provider_id`) → **codificación por destino/frecuencia con *cross-fitting***, **dentro del `Pipeline`** (jamás ajustada con todo el dataset: sería fuga).
2. **Cuida la disponibilidad temporal.** Razona si `claim_status`/`approved_amount` son anteriores o posteriores a la predicción; documenta tu decisión.
3. **Partición estratificada**; el *test* solo se toca al final.
4. **Sin fuga en el preprocesado:** fechas, imputación, codificación y remuestreo dentro del `Pipeline`, reajustados en cada pliegue.
5. **Equilibrado solo en *train***; el fraude es minoritario.
6. **Marco operativo:** además de clasificar, **prioriza** (ordena por probabilidad y mide precision@K / curva de ganancia): un equipo de auditoría solo revisa un número limitado de casos.
7. **Reproducibilidad y honestidad:** `random_state` fijado; reporta lo que no funcionó.

# <font color="steelblue">Fase 0 — Preparación del entorno y carga de datos</font>

In [ ]:
# !pip -q install kagglehub imbalanced-learn category_encoders scikit-learn shap gradio
import os, warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings('ignore'); sns.set_theme(style='whitegrid')
import kagglehub
RNG = 42

In [ ]:
# Paso 1: descargar el dataset
path = kagglehub.dataset_download("nudratabbas/healthcare-fraud-detection-dataset")
print("Ruta al dataset:", path)
archivos = os.listdir(path)
print("Archivos:", archivos)

# Paso 2: cargar el CSV principal (ajusta el nombre si difiere)
healthcare_fraud = pd.read_csv(os.path.join(path, archivos[0]))
print(f"Dimensiones: {healthcare_fraud.shape[0]:,} filas × {healthcare_fraud.shape[1]} columnas")
healthcare_fraud.head()

# <font color="steelblue">Fase 1 — Comprensión y EDA</font>

**Tareas obligatorias**
1. **Objetivo.** Distribución de `Is_Fraud` (¿qué % de fraude? ¿cómo de desequilibrado?).
2. **Tipos y cardinalidad (clave).** Para cada categórica, contad **valores únicos**: identificad las de **alta cardinalidad** (`diagnosis_code`, `procedure_code`, `state`, `provider_id`) frente a las de baja (`insurance_type`, `visit_type`, `claim_status`, `provider_specialty`, `patient_gender`).
3. **Faltantes.** Porcentaje por columna y plan de imputación.
4. **Fechas.** Rango de `claim_submission_date`; relación de `days_to_submission` con el fraude.
5. **Señales de diseño.** Comprobad los patrones esperados: fraude ↔ `claim_amount` alto, `days_to_submission` corto, concentración por `provider_id`.
6. **Disponibilidad temporal.** Discutid `claim_status`/`approved_amount` (¿posteriores a la decisión?).
7. **Conclusión:** 3–4 hallazgos.

> **A responder:** con fraude minoritario y coste asimétrico, ¿qué métricas usaréis? (pista: **recall** de fraude, **PR-AUC**, precision@K; la *accuracy* no vale).

# <font color="steelblue">Fase 2 — Ingeniería de variables, codificación SIN fuga y partición</font>

Este es el **núcleo** del proyecto.

**2A. Ingeniería de variables**
1. **Fechas:** de `claim_submission_date` derivad mes, día de la semana, etc. (y decidid si `claim_submission_date` cruda se descarta).
2. (Opcional) ratios con sentido: `approved_amount/claim_amount`, etc. — pero **cuidado con la fuga** si esas variables son posteriores.

**2B. Codificación por cardinalidad (clave)**
3. **Baja cardinalidad** → `OneHotEncoder` (con `min_frequency`/`handle_unknown` para agrupar raras y tolerar categorías nuevas).
4. **Alta cardinalidad** (`diagnosis_code`, `procedure_code`, `state`, `provider_id`) → **codificación por destino** (`sklearn.preprocessing.TargetEncoder`, que hace *cross-fitting* interno) o por **frecuencia**. **Debe ir dentro del `Pipeline`** para que se ajuste **solo con el *train*** de cada pliegue.
5. **Numéricas:** imputación + escalado (para logística/SVM/kNN; los árboles no lo necesitan). `claim_amount` log-normal → valorad `log1p`.

**2C. Partición y `Pipeline`**
6. `X`/`y`, **partición estratificada** y `ColumnTransformer`/`Pipeline` que aplique cada bloque a su grupo de columnas.

> **A responder:** ¿por qué codificar `provider_id` por su tasa de fraude usando **todo** el dataset es una fuga grave? ¿Cómo lo evita el *cross-fitting*?

# <font color="steelblue">Fase 3 — Modelos base y comparación</font>

**Tareas obligatorias**
1. Comparad **≥5 familias** del curso: **Regresión logística**, **kNN**, **SVM**, **Árbol**, **Random Forest**, **HistGradientBoosting** (o XGBoost/LightGBM/CatBoost — muy adecuados para tabular mixto), **Naive Bayes**.
2. **CV estratificada** con una métrica robusta al desequilibrio (**`average_precision`**/PR-AUC o **recall** de fraude).
3. **Tabla** comparativa + comentario (los **boosting de árboles** suelen brillar en tabular con categóricas).

# <font color="steelblue">Fase 4 — Ponderación de muestras y equilibrado</font>

El fraude es **minoritario**: tratad el desequilibrio (material **11**) sobre los 2–3 mejores modelos:

1. **Sin tratamiento** (línea base).
2. **Sensible al coste:** `class_weight='balanced'` (o `sample_weight`).
3. **Sobremuestreo:** **SMOTE**/**SMOTENC** (¡tenéis categóricas codificadas! usad SMOTENC o aplicad SMOTE tras la codificación numérica, dentro del `ImbPipeline`).
4. (Opcional) submuestreo/híbrido.

Reportad **recall de fraude**, **F1**, **PR-AUC** y exactitud balanceada, y razonad la mejor opción dado el **coste** de un fraude no detectado.

> **Sin fugas:** remuestreo dentro de `Pipeline` de *imbalanced-learn*, solo en *train*.

# <font color="steelblue">Fase 5 — Optimización de hiperparámetros</font>

1. Optimizad los **2–3 mejores** (modelo + equilibrado).
2. `GridSearchCV`/`RandomizedSearchCV`/**Optuna**, con CV estratificada y la métrica elegida (PR-AUC/recall); búsqueda **sobre el `Pipeline`** (prefijo `clf__`); podéis incluir parámetros del **codificador** (p. ej. la suavización del `TargetEncoder`).
3. Reportad mejores hiperparámetros y la mejora.

# <font color="steelblue">Fase 6 — Combinación de modelos</font>

1. Combinad los mejores con **`VotingClassifier`** (votación **blanda**) y/o **`StackingClassifier`**.
2. Comparad frente al **mejor individual** (PR-AUC/recall): ¿mejora? ¿compensa el coste?
3. **Combinad solo si aporta** mejora real (requisito: *si fuera necesario*).

# <font color="steelblue">Fase 7 — Evaluación, priorización de investigaciones e interpretación</font>

El *test* se usa una sola vez.

**Tareas obligatorias**
1. **Métricas finales:** **matriz de confusión**, **recall de fraude**, **precisión**, **F1**, **ROC-AUC** y **PR-AUC**.
2. **Priorización (marco operativo, clave).** Un equipo solo audita un número limitado de casos. Ordenad las reclamaciones por **probabilidad de fraude** y calculad:
   * **precision@K** y la **curva de ganancia** (cuántos fraudes reales se capturan revisando los K primeros),
   * un **umbral por coste** (coste de auditar vs pérdida por fraude no detectado).
   Es la misma idea que en el cuaderno de *Detección de anomalías*.
3. **Interpretabilidad (SHAP).** ¿Pesan `claim_amount`, `days_to_submission`, el proveedor…, como anticipan los patrones de diseño? ¿Coincide?
4. **Discusión crítica:** datos **sintéticos** (¿generaliza?), riesgo de penalizar injustamente a proveedores, sesgos, y el papel del modelo como **filtro** que prioriza, no como juez.

# <font color="steelblue">Fase 8 — Despliegue del modelo</font>

1. **Persistencia:** guardad el **`Pipeline` completo** (ingeniería + codificación + modelo) con `joblib`; debe aceptar una reclamación **cruda**.
2. **Función de *scoring*:** `puntuar_reclamacion(...)` que devuelva la **probabilidad de fraude** y una **recomendación** (investigar / aprobar) según el umbral elegido.
3. **Interfaz / cola de auditoría:** app **Gradio** que puntúe una reclamación (o suba un CSV y devuelva el **ranking** de las más sospechosas para el equipo de auditoría). En Colab da un **enlace público**.
4. (Opcional, nota extra) **Streamlit**/**FastAPI**, o combinar con **Isolation Forest** (no supervisado) como segundo filtro.

> **Aviso (obligatorio):** herramienta **educativa** de **priorización**; señala casos a **revisar**, no determina culpabilidad. Datos sintéticos.

# <font color="steelblue">Pistas y errores típicos</font>

* **Codificación con fuga.** Calcular la tasa de fraude por `provider_id`/`diagnosis_code` con todo el dataset y usarla como variable es fuga directa: usa **`TargetEncoder`** (cross-fitting) **dentro** del `Pipeline`.
* **One-hot no escala** con ICD-10/CPT: agrupa raras (`min_frequency`) o usa codificación por destino/frecuencia.
* **¿Variables del futuro?** `claim_status`/`approved_amount` pueden ser posteriores a la decisión; razónalo.
* **No mires solo la *accuracy*.** Con fraude minoritario, mira **recall de fraude**, **PR-AUC** y, sobre todo, **precision@K** (lo que de verdad importa para auditar).
* **Marco operativo:** entrega un **ranking** de casos a investigar, no solo una etiqueta.
* **Despliegue:** guarda el **Pipeline entero** y respeta el orden/formato de las columnas.

# <font color="steelblue">Referencias</font>

* *Healthcare Fraud Detection Dataset* (Kaggle, *nudratabbas*). Datos sintéticos.
* Estándares de codificación médica: **ICD-10** (diagnósticos) y **CPT** (procedimientos).
* Cuadernos del curso: *Boosting*, *Random Forest*, *Detección de anomalías (Isolation Forest)*, *Equilibrando las muestras*, *Regresión logística binaria*.